# Análisis de Supervivencia: Curvas de Kaplan-Meier

En este cuaderno exploramos la dinámica temporal del egreso en Matemáticas y Física. A diferencia del EDA estático, el análisis de supervivencia nos permite tratar correctamente a los alumnos que aún no terminan (datos censurados).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter
from pathlib import Path

# Configuración
DATOS_DIR = Path('../../Datos/Datos_limpios_activos')
sns.set_theme(style="whitegrid")

In [ ]:
def load_survival_data(carrera):
    df = pd.read_excel(DATOS_DIR / f'{carrera}.xlsx')
    # Definir T (tiempo) y E (evento)
    # Si semestre_termino es NaN o 20, consideramos que el evento NO ha ocurrido (censura)
    df['T'] = df['semestre_termino'].fillna(20)
    df['E'] = (df['semestre_termino'] < 20).astype(int)
    df['Carrera'] = carrera
    return df

df_mat = load_survival_data('Matemáticas')
df_fis = load_survival_data('Física')
df_all = pd.concat([df_mat, df_fis])

In [ ]:
kmf = KaplanMeierFitter()

plt.figure(figsize=(10, 6))

for carrera in ['Matemáticas', 'Física']:
    data = df_all[df_all['Carrera'] == carrera]
    kmf.fit(data['T'], event_observed=data['E'], label=carrera)
    kmf.plot_survival_function()

plt.title('Curvas de Kaplan-Meier: Probabilidad de seguir sin titularse', fontsize=15)
plt.xlabel('Semestres Transcurridos')
plt.ylabel('Probabilidad de No-Titulación')
plt.axvline(x=8, color='blue', linestyle='--', alpha=0.5, label='Ideal Mate (S8)')
plt.axvline(x=9, color='red', linestyle='--', alpha=0.5, label='Ideal Física (S9)')
plt.legend()
plt.show()

## Mediana de Tiempo de Egreso
La mediana de supervivencia es el tiempo en el que el 50% de la población ha experimentado el evento (titulación).

In [ ]:

# Calcular e imprimir las Medianas e Intervalos de Confianza (95%)
print("--- Mediana de Sobrevivencia (Titulación) ---")
for carrera in ['Matemáticas', 'Física']:
    data = df_all[df_all['Carrera'] == carrera]
    kmf.fit(data['T'], event_observed=data['E'])
    mediana = kmf.median_survival_time_
    
    # Extraer el intervalo de confianza de la mediana
    from lifelines.utils import median_survival_times
    median_ci = median_survival_times(kmf.confidence_interval_)
    
    lower = median_ci.iloc[0, 0]
    upper = median_ci.iloc[0, 1]
    
    print(f"[{carrera}] Mediana: {mediana} semestres. CI 95%: [{lower}, {upper}]")



## Prueba de Hipótesis: Log-Rank Test
Para concluir con rigor estadístico si la diferencia entre las curvas de supervivencia de Física y Matemáticas es significativa, aplicaremos el Log-Rank Test. Si el p-valor es menor a 0.05, rechazamos la hipótesis nula de que ambas poblaciones tienen la misma probabilidad de supervivencia.


In [ ]:

from lifelines.statistics import logrank_test

df_mat = df_all[df_all['Carrera'] == 'Matemáticas']
df_fis = df_all[df_all['Carrera'] == 'Física']

results = logrank_test(df_mat['T'], df_fis['T'], event_observed_A=df_mat['E'], event_observed_B=df_fis['E'])

print("--- Log-Rank Test: Matemáticas vs Física ---")
print(f"Estadístico de prueba: {results.test_statistic:.2f}")
print(f"P-valor: {results.p_value:.4e}")

if results.p_value < 0.05:
    print("Conclusión: Existe una diferencia estadísticamente significativa entre Matemáticas y Física.")
else:
    print("Conclusión: No hay evidencia suficiente para afirmar una diferencia significativa.")



## Análisis de Perfiles: La Paradoja del Perfeccionismo
Segmentaremos a los alumnos al 4to semestre según la mediana de su Regularidad y Promedio Natural.
Luego, ajustaremos un Modelo de Riesgos Proporcionales de Cox para cuantificar el impacto de cada variable.


In [ ]:

from lifelines import CoxPHFitter
from lifelines.statistics import multivariate_logrank_test

med_reg = df_all['s4_reg'].median()
med_nat = df_all['s4_nat'].median()

def get_profile(row):
    if row['s4_reg'] >= med_reg and row['s4_nat'] >= med_nat: return 'Élite'
    if row['s4_reg'] >= med_reg and row['s4_nat'] < med_nat: return 'Pragmático'
    if row['s4_reg'] < med_reg and row['s4_nat'] >= med_nat: return 'Perfeccionista'
    return 'Rezago'

df_clean = df_all.dropna(subset=['s4_reg', 's4_nat']).copy()
df_clean['Perfil'] = df_clean.apply(get_profile, axis=1)

# Plot Kaplan-Meier para los perfiles
plt.figure(figsize=(10, 6))
for perfil in df_clean['Perfil'].unique():
    subset = df_clean[df_clean['Perfil'] == perfil]
    kmf.fit(subset['T'], event_observed=subset['E'], label=perfil)
    kmf.plot_survival_function()
plt.title('Supervivencia por Perfil Académico (al S4)')
plt.show()

# Cox Model
cph = CoxPHFitter()
cph.fit(df_clean[['T', 'E', 's4_reg', 's4_nat']], duration_col='T', event_col='E')
cph.print_summary()
